In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
# Global plot style
matplotlib.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 16,
    'axes.labelweight': 'medium',
    'xtick.labelsize': 14,
    'ytick.labelsize': 14
})


In [ ]:
# Paths and output directory
DATA_PATH = "../1_data/processed/zone_sequence_merged.csv"
SAVE_DIR = "plots/eda_fire"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Load merged dataset
climate_fire_df = pd.read_csv(DATA_PATH)
climate_fire_df["Date"] = pd.to_datetime(climate_fire_df["Date"])
climate_fire_df["Month"] = climate_fire_df["Date"].dt.month
climate_fire_df["Year"] = climate_fire_df["Date"].dt.year

In [ ]:
# Aggregate region-level daily data
region_daily_df = climate_fire_df.groupby("Date").agg({
    "Num_Fires": "sum",
    "Temperature": "mean",
    "Humidity": "mean",
    "Precipitation": "mean",
    "Wind": "mean"
}).reset_index()
region_daily_df["Month"] = region_daily_df["Date"].dt.month
region_daily_df["Year"] = region_daily_df["Date"].dt.year

In [ ]:
# Daily fire count distribution per zone
zones = sorted(climate_fire_df["Zone_ID"].unique())
max_class = int(climate_fire_df["Num_Fires"].max())
max_count = max(climate_fire_df[climate_fire_df["Zone_ID"] == z]["Num_Fires"].value_counts().max() for z in zones)

fig, axes = plt.subplots(2, 4, figsize=(22, 12), sharex=True, sharey=True)
axes = axes.flatten()

for i, zone in enumerate(zones):
    zone_df = climate_fire_df[climate_fire_df["Zone_ID"] == zone]
    fire_counts = zone_df["Num_Fires"].value_counts().sort_index()
    axes[i].bar(fire_counts.index, fire_counts.values, color="steelblue", edgecolor="black", linewidth=0.5)
    axes[i].set_title(f"Zone {zone}", fontsize=20, fontweight='bold', pad=10)
    axes[i].set_ylim(0, max_count * 1.1)
    axes[i].set_xlim(-0.5, max_class + 0.5)
    axes[i].set_xticks(range(0, max_class + 1, 2))
    axes[i].grid(axis="y", linestyle="--", alpha=0.6)

fig.text(0.5, 0.04, "Number of Fires", ha="center", fontsize=20, fontweight='bold')
fig.text(0.03, 0.5, "Number of Days", ha="center", va="center", rotation="vertical", fontsize=20, fontweight='bold')
plt.subplots_adjust(left=0.07, bottom=0.10, right=0.98, top=0.93, wspace=0.10, hspace=0.25)
plt.savefig(os.path.join(SAVE_DIR, "daily_fire_count_distribution_zones.png"), dpi=600, bbox_inches="tight")
plt.close()

In [ ]:
# Daily fire count distribution for the region
counts = region_daily_df["Num_Fires"].value_counts().sort_index()
max_class_r = int(region_daily_df["Num_Fires"].max())
max_count_r = int(counts.max())

fig, ax = plt.subplots(figsize=(22, 12))
ax.bar(counts.index, counts.values, color="steelblue", edgecolor="black", linewidth=0.5)
ax.set_xlim(-0.5, max_class_r + 0.5)
ax.set_ylim(0, max_count_r * 1.1)
ax.set_xticks(range(0, max_class_r + 1, 2))
ax.grid(axis="y", linestyle="--", alpha=0.6)

fig.text(0.5, 0.03, "Number of Fires", ha="center", fontsize=20, fontweight='bold')
fig.text(0.02, 0.5, "Number of Days", ha="center", va="center", rotation="vertical", fontsize=20, fontweight='bold')
plt.subplots_adjust(left=0.06, bottom=0.10, right=0.98, top=0.93)
plt.savefig(os.path.join(SAVE_DIR, "daily_fire_count_distribution_region.png"), dpi=600, bbox_inches="tight")
plt.close()

In [ ]:
# Monthly fire totals per zone
months = [3, 4, 5, 6, 7, 8, 9, 10]
month_labels = ["Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct"]

monthly = (climate_fire_df.groupby(["Zone_ID", "Month"])["Num_Fires"]
           .sum().unstack(fill_value=0).reindex(columns=months, fill_value=0))
max_y = monthly.values.max()

fig, axes = plt.subplots(2, 4, figsize=(22, 12), sharex=True, sharey=True)
axes = axes.flatten()

for i, z in enumerate(zones):
    vals = monthly.loc[z].values
    axes[i].bar(range(len(months)), vals, color="steelblue", edgecolor="black", linewidth=0.5)
    axes[i].set_title(f"Zone {z}", fontsize=20, fontweight='bold', pad=10)
    axes[i].set_ylim(0, max_y * 1.1)
    axes[i].set_xticks(range(len(months)))
    axes[i].set_xticklabels(month_labels, fontsize=14)
    axes[i].grid(axis="y", linestyle="--", alpha=0.6)

fig.text(0.5, 0.04, "Month", ha="center", fontsize=20, fontweight='bold')
fig.text(0.03, 0.5, "Number of Fires", ha="center", va="center", rotation="vertical", fontsize=20, fontweight='bold')
plt.subplots_adjust(left=0.07, bottom=0.10, right=0.98, top=0.93, wspace=0.10, hspace=0.25)
plt.savefig(os.path.join(SAVE_DIR, "monthly_fire_totals_zones.png"), dpi=600, bbox_inches="tight")
plt.close()

In [ ]:
# Monthly fire totals for the region
monthly_regional = (region_daily_df[region_daily_df["Month"].isin(months)]
                    .groupby("Month")["Num_Fires"].sum()
                    .reindex(months, fill_value=0))

fig, ax = plt.subplots(figsize=(22, 12))
ax.bar(range(len(months)), monthly_regional.values, color="steelblue", edgecolor="black", linewidth=0.5)
ax.set_ylim(0, monthly_regional.max() * 1.1)
ax.set_xticks(range(len(months)))
ax.set_xticklabels(month_labels, fontsize=16)
ax.grid(axis="y", linestyle="--", alpha=0.6)

fig.text(0.5, 0.03, "Month", ha="center", fontsize=20, fontweight='bold')
fig.text(0.02, 0.5, "Number of Fires", ha="center", va="center", rotation="vertical", fontsize=20, fontweight='bold')
plt.subplots_adjust(left=0.07, bottom=0.10, right=0.98, top=0.93)
plt.savefig(os.path.join(SAVE_DIR, "monthly_fire_totals_region.png"), dpi=600, bbox_inches="tight")
plt.close()

In [ ]:
# Yearly fire totals per zone
years = list(range(2008, 2019))
yearly = (climate_fire_df.groupby(["Zone_ID", "Year"])["Num_Fires"]
          .sum().unstack(fill_value=0).reindex(columns=years, fill_value=0))
max_y = yearly.values.max()

fig, axes = plt.subplots(2, 4, figsize=(22, 12), sharex=True, sharey=True)
axes = axes.flatten()

for i, z in enumerate(zones):
    vals = yearly.loc[z].values
    axes[i].bar(years, vals, color="steelblue", edgecolor="black", linewidth=0.5)
    axes[i].set_title(f"Zone {z}", fontsize=20, fontweight='bold', pad=10)
    axes[i].set_ylim(0, max_y * 1.1)
    axes[i].set_xticks(years)
    axes[i].set_xticklabels(years, rotation=45, fontsize=14)
    axes[i].grid(axis="y", linestyle="--", alpha=0.6)

fig.text(0.5, 0.04, "Year", ha="center", fontsize=20, fontweight='bold')
fig.text(0.03, 0.5, "Number of Fires", ha="center", va="center", rotation="vertical", fontsize=20, fontweight='bold')
plt.subplots_adjust(left=0.07, bottom=0.12, right=0.98, top=0.93, wspace=0.10, hspace=0.30)
plt.savefig(os.path.join(SAVE_DIR, "yearly_fire_totals_zones.png"), dpi=600, bbox_inches="tight")
plt.close()

In [ ]:
# Yearly fire totals for the region
yearly_regional = (region_daily_df.groupby("Year")["Num_Fires"]
                   .sum().reindex(range(2008, 2019), fill_value=0))

fig, ax = plt.subplots(figsize=(22, 12))
ax.bar(years, yearly_regional.values, color="steelblue", edgecolor="black", linewidth=0.5)
ax.set_ylim(0, yearly_regional.max() * 1.1)
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, fontsize=16)
ax.grid(axis="y", linestyle="--", alpha=0.6)

fig.text(0.5, 0.06, "Year", ha="center", fontsize=20, fontweight='bold')
fig.text(0.03, 0.5, "Number of Fires", ha="center", va="center", rotation="vertical", fontsize=20, fontweight='bold')
plt.subplots_adjust(left=0.07, bottom=0.15, right=0.98, top=0.93)
plt.savefig(os.path.join(SAVE_DIR, "yearly_fire_totals_region.png"), dpi=600, bbox_inches="tight")
plt.close()


In [ ]:
# Daily fire time series per zone (fire season only, no gaps)
max_y = climate_fire_df["Num_Fires"].max()

fig, axes = plt.subplots(2, 4, figsize=(22, 14), sharey=True)
axes = axes.flatten()

for i, z in enumerate(zones):
    zone_df = climate_fire_df[climate_fire_df["Zone_ID"] == z].copy()
    zone_df = zone_df[zone_df["Month"].between(3, 10)].sort_values("Date").reset_index(drop=True)

    axes[i].plot(range(len(zone_df)), zone_df["Num_Fires"],
                 color="#1f77b4", linewidth=1.5, marker="o", markersize=3)
    axes[i].set_title(f"Zone {z}", fontsize=20, fontweight='bold', pad=10)
    axes[i].set_ylim(0, max_y * 1.1)

    xticks = np.arange(0, len(zone_df), 365)
    xtick_labels = zone_df["Date"].dt.strftime('%b %Y').iloc[xticks]
    axes[i].set_xticks(xticks)
    axes[i].set_xticklabels(xtick_labels, rotation=45, fontsize=14)
    axes[i].grid(axis="y", linestyle="--", alpha=0.6)
    for xt in xticks:
        axes[i].axvline(x=xt, color="gray", linestyle="--", alpha=0.4, linewidth=0.8)

fig.text(0.5, 0.04, "Day in Fire Season", ha="center", fontsize=20, fontweight='bold')
fig.text(0.03, 0.5, "Number of Fires", ha="center", va="center", rotation="vertical", fontsize=20, fontweight='bold')
plt.subplots_adjust(left=0.07, bottom=0.14, right=0.98, top=0.93, wspace=0.10, hspace=0.45)
plt.savefig(os.path.join(SAVE_DIR, "daily_fire_time_series_zones.png"), dpi=600, bbox_inches="tight")
plt.close()

In [ ]:
# Daily fire time series for the region (fire season only, no gaps)
region_fire_season_df = region_daily_df[region_daily_df["Month"].between(3, 10)].sort_values("Date").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(24, 14))
ax.plot(range(len(region_fire_season_df)), region_fire_season_df["Num_Fires"],
        color="#1f77b4", linewidth=1.5, marker="o", markersize=3)
ax.set_ylim(0, region_fire_season_df["Num_Fires"].max() * 1.1)

xticks = np.arange(0, len(region_fire_season_df), 90)
xtick_labels = region_fire_season_df["Date"].dt.strftime('%b %Y').iloc[xticks]
ax.set_xticks(xticks)
ax.set_xticklabels(xtick_labels, rotation=45, fontsize=14)
ax.grid(axis="y", linestyle="--", alpha=0.6)
for xt in xticks:
    ax.axvline(x=xt, color="gray", linestyle="--", alpha=0.4, linewidth=0.8)

fig.text(0.5, 0.07, "Day in Fire Season", ha="center", fontsize=20, fontweight='bold')
fig.text(0.05, 0.5, "Number of Fires", ha="center", va="center", rotation="vertical", fontsize=20, fontweight='bold')
plt.subplots_adjust(left=0.09, bottom=0.18, right=0.98, top=0.93)
plt.savefig(os.path.join(SAVE_DIR, "daily_fire_time_series_region.png"), dpi=600, bbox_inches="tight")
plt.close()